In [ ]:
import numpy as np
import networkx as nx
from basic_neat import DAG, plot_dag, old_plot_dag, NEAT
from basic_neat import ReLU, LeakyReLU, Sigmoid, Linear, Tanh, Softmax, Adam, CrossEntropy, MSE

def dummy_uuid_generator():
    i = 0
    while True:
        yield str(i)
        i += 1

dummy_gen = dummy_uuid_generator()

def uuid1():
    return next(dummy_gen)

In [ ]:
INPUTS = 2
OUTPUTS = 1

# test creation of DAG
dag = DAG(INPUTS, OUTPUTS, floating_outputs=True)
plot_dag(dag)

In [ ]:
# test adding connections
for n1_id in dag.input_nodes:
    for n2_id in dag.output_nodes:
        n1 = dag.get_node(n1_id)
        n2 = dag.get_node(n2_id)
        dag.add_connection(n1.id, n2.id)
plot_dag(dag)

In [ ]:
# test getting conections and connection check
connections = dag.get_connections()
print("Created Connections", connections)
print("Check if connection can be found:", dag.is_connected(dag.input_nodes[0], dag.output_nodes[0]))

In [ ]:
# test adding nodes
for connection in connections:
    print(f"Adding Node between {(connection[0], connection[1])}", dag.add_node(connection[0], connection[1]))
plot_dag(dag)

In [ ]:
dag.add_connection('3', '4')
 
plot_dag(dag)

In [ ]:
dag.add_node('3', '4')
 
plot_dag(dag)

In [ ]:
dag.add_node('0', '3')
 
plot_dag(dag)

In [ ]:
 # test getting processing order
print("Processing Order:", dag.get_processing_order())
 
plot_dag(dag)

In [ ]:
print("Starting Cons:", dag.get_connections())

dag.remove_node('3')
    

print("Ending Cons:", dag.get_connections())
 
plot_dag(dag)

In [ ]:
dag.kill_floaters()
plot_dag(dag)

In [ ]:
dag.add_connection('0', '4')
ok, new_id = dag.add_node('0', '4')
dag.add_connection('4', new_id)
dag.add_node('4','2')
old_plot_dag(dag)

In [ ]:
dag.remove_connection('4', '7')
old_plot_dag(dag)

# Mutation Test

In [ ]:
for i in range(10):
    dag.mutate()   
    plot_dag(dag)

In [ ]:
print(dag.get_legal_connections())

# Mini Sanity Test

In [ ]:
import matplotlib.pyplot as plt

# 1D Testdaten
X = np.linspace(-2, 2, 200).reshape(-1, 1)
Y = np.sin(X)   # Ziel: Sinus lernen

# Netz aufbauen
dag = DAG(1, 1, fully_connect=True, standard_output_activation=Tanh())
# Manuell Hidden-Node einfügen (optional)
for _ in range(3):
    con = dag.get_connections()[0]
    dag.add_node(con[0], con[1], ReLU())

# Training
losses = dag.train(X, Y, epochs=200, batch_size=32, verbose=True)

# Loss plotten
plt.plot(losses)
plt.title("Training Loss")
plt.show()

# Predictions plotten
Y_pred = dag.process(X)
plt.plot(X, Y, label="Target")
plt.plot(X, Y_pred, label="Prediction")
plt.legend()
plt.show()

# XOR problem

In [ ]:
X = [[0,0],[0,1],[1,0],[1,1]]
Y = [[0],[1],[1],[0]]  # regression style in [-1,1] works better with tanh, so map to {-1,1}
Y_tanh = [[-1],[1],[1],[-1]]

dag = DAG(nr_inputs=2, nr_outputs=1, fully_connect=True, standard_output_activation=Linear())
# add 2 hidden nodes
for i in range(2):
    # add connection from input to output
    dag.add_connection(dag.input_nodes[0], dag.output_nodes[0])
    ok, new_node_id = dag.add_node(dag.input_nodes[0], dag.output_nodes[0], ReLU())

    # connect to all remaining input and output nodes
    for inp in dag.input_nodes[1:]:
        dag.add_connection(inp, new_node_id)
    for out in dag.output_nodes[1:]:
        dag.add_connection(new_node_id, out)

print(dag)
plot_dag(dag)

In [ ]:
losses = dag.train(X, Y, epochs=1000, batch_size=4, verbose=True)
# Loss plotten
plt.plot(losses)
plt.title("Training Loss")
plt.show()

# Predictions plotten
Y_pred = dag.process(X)
for x, y in zip(X, Y_pred):
    print(f"Input: {x}, Pred: {y}, Target: {Y[X.index(x)]}")
plt.scatter(range(len(Y)), Y, label="Target")
plt.scatter(range(len(Y_pred)), Y_pred, label="Prediction")
plt.legend()
plt.show()

plot_dag(dag)

# test space

In [ ]:
from keras.datasets import mnist, boston_housing
import matplotlib.pyplot as plt
# Load the MNIST dataset
""" (x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train = x_train.reshape(-1, 28*28) / 255
x_test = x_test.reshape(-1, 28*28) / 255

# convert Y to one-hot encoding
y_train = np.eye(10)[y_train] """

# -------------------
# Daten vorbereiten
# -------------------
(x_train, y_train), (x_test, y_test) = boston_housing.load_data()

# normalize features
x_mean, x_std = np.mean(x_train, axis=0), np.std(x_train, axis=0)
x_train = (x_train - x_mean) / x_std
x_test  = (x_test  - x_mean) / x_std

# normalize targets
y_max = np.max(y_train)
y_train = y_train.reshape(-1, 1) / y_max
y_test  = y_test.reshape(-1, 1) / y_max

In [ ]:
boston_dag = DAG(x_train.shape[1], y_train.shape[1], fully_connect=True, standard_output_activation=Sigmoid())

# add 3 hidden nodes
for i in range(3):
    # add connection from input to output
    boston_dag.add_connection(boston_dag.input_nodes[0], boston_dag.output_nodes[0])
    ok, new_node_id = boston_dag.add_node(boston_dag.input_nodes[0], boston_dag.output_nodes[0], ReLU())

    print("New Node ID:", new_node_id)

    # connect to all remaining input and output nodes
    for inp in boston_dag.input_nodes[1:]:
        boston_dag.add_connection(inp, new_node_id)
    for out in boston_dag.output_nodes[1:]:
        boston_dag.add_connection(new_node_id, out)

plot_dag(boston_dag)

In [ ]:
losses = boston_dag.train(x_train, y_train, epochs=200, verbose=True)

In [ ]:
# plot losses
print("Losses:", losses)
plt.plot(losses)
plt.show()

In [ ]:
yHat = boston_dag.process(x_train)
for i in range(10):
    print("Pred:", np.around(yHat[i], 3), "Target:", y_train[i], "Error:", y_train[i] - yHat[i])

In [ ]:
for i in range(10):
    #plt.imshow(x_test[i].reshape(28, 28))
    #plt.show()
    
    output = boston_dag.process([x_train[i]])
    print("YHat:", output, "Y:", y_train[i])

# MNIST

In [ ]:
(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train = x_train.reshape(-1, 28*28) / 255
x_test = x_test.reshape(-1, 28*28) / 255

# convert Y to one-hot encoding
y_train = np.eye(10)[y_train]
y_test = np.eye(10)[y_test]

In [ ]:
mnist_dag = DAG(28*28, 10, fully_connect=True, standard_output_activation=Sigmoid(), optimizer=Adam(lr=1e-3), error_function=MSE())
# add 10 hidden nodes
for i in range(10):
    # add connection from input to output
    mnist_dag.add_connection(mnist_dag.input_nodes[0], mnist_dag.output_nodes[0])
    ok, new_node_id = mnist_dag.add_node(mnist_dag.input_nodes[0], mnist_dag.output_nodes[0], ReLU())

    print("New Node ID:", new_node_id)

    # connect to all remaining input and output nodes
    for inp in mnist_dag.input_nodes[1:]:
        mnist_dag.add_connection(inp, new_node_id)
    for out in mnist_dag.output_nodes[1:]:
        mnist_dag.add_connection(new_node_id, out)


In [ ]:
print(x_train.shape, y_train.shape)
losses = mnist_dag.train(x_train, y_train, epochs=2, verbose=True)

# plot losses
print("Losses:", losses)
plt.plot(losses)
plt.show()

In [ ]:
# Predictions plotten
yHat = mnist_dag.process(x_train)
for i in range(10):
    print("Pred:", np.around(yHat[i], 3), "Target:", y_train[i], "Pred Label:", np.argmax(yHat[i]), "Target Label:", np.argmax(y_train[i]))
    #plt.imshow(x_test[i].reshape(28, 28))
    #plt.show()

# NEAT test

In [ ]:
# -------------------
# Daten vorbereiten
# -------------------
(x_train, y_train), (x_test, y_test) = boston_housing.load_data()

# normalize features
x_mean, x_std = np.mean(x_train, axis=0), np.std(x_train, axis=0)
x_train = (x_train - x_mean) / x_std
x_test  = (x_test  - x_mean) / x_std

# normalize targets
y_max = np.max(y_train)
y_train = y_train.reshape(-1, 1) / y_max
y_test  = y_test.reshape(-1, 1) / y_max

In [ ]:
neat_env = NEAT(nr_inputs=x_train.shape[1], nr_outputs=y_train.shape[1], 
                population_size=5,
               output_activation=Sigmoid(), optimizer=Adam(lr=1e-3), error_function=MSE())

In [ ]:
best_of_generations = neat_env.run(x_train, y_train, generations=3, epochs=200, batch_size=32, verbose=False)

In [ ]:
print(f"Best of 1st Generations loss: {best_of_generations[0][1]}, acc: {best_of_generations[0][2]}", )
plot_dag(best_of_generations[0][0])
print(f"Best of last Generations loss: {best_of_generations[-1][1]}, acc: {best_of_generations[-1][2]}", )
plot_dag(best_of_generations[-1][0])

# Test resulting DAG

In [ ]:
best_dag = best_of_generations[-1][0]
new_dag = DAG.from_other(best_dag)
plot_dag(new_dag)

In [ ]:
losses = new_dag.train(x_train, y_train, epochs=100, batch_size=32, verbose=True)
# Loss plotten
plt.plot(losses)
plt.title("Training Loss")
plt.show()

In [ ]:
for i in range(10):
    #plt.imshow(x_test[i].reshape(28, 28))
    #plt.show()

    output = new_dag.process([x_train[i]])
    print("YHat:", output, "Y:", y_train[i])

# NEAT MNIST Test

In [ ]:
(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train = x_train.reshape(-1, 28*28) / 255
x_test = x_test.reshape(-1, 28*28) / 255

# convert Y to one-hot encoding
y_train = np.eye(10)[y_train]
y_test = np.eye(10)[y_test]

In [ ]:
neat_env = NEAT(nr_inputs=x_train.shape[1], nr_outputs=y_train.shape[1], 
                population_size=5,
               output_activation=Sigmoid(), optimizer=Adam(lr=1e-3), error_function=MSE())

In [ ]:
best_of_generations = neat_env.run(x_train, y_train, generations=3, epochs=2, batch_size=32, verbose=False)

In [ ]:
print(f"Best of 1st Generations loss: {best_of_generations[0][1]}, acc: {best_of_generations[0][2]}")
print(f"Best of last Generations loss: {best_of_generations[-1][1]}, acc: {best_of_generations[-1][2]}")

In [ ]:
best_dag = best_of_generations[-1][0]
yHat = best_dag.process(x_train)
for i in range(10):
    print("Pred:", np.around(yHat[i], 3), "Target:", y_train[i], "Pred Label:", np.argmax(yHat[i]), "Target Label:", np.argmax(y_train[i]))